# Lab 9 — Let an AI attack your project
### EP0418 · Topic 9 · 1 practical hour

**You need:** your Lab 8 notebook (in your Google Drive) and the printed Lab 9
sheet. This notebook fetches its own files.

---
Today you point an AI assistant at your own pipeline and ask it to **find what is
wrong with it**.

> **What is assessed is your rebuttal, not what the critic finds.**

The critic will raise things that are wrong. Saying *why* they are wrong, with
evidence, is worth exactly as much as fixing the ones that are right.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

> **Then run the grey code cell just below** (click in it and press Shift+Enter, or click the play button at its left). Colab may warn that the notebook was not authored by Google: click **Run anyway**. You should see nothing in red; lines like *fetched ...*, a row count or a short table are normal. That means the course files and tools are loaded. Every lab starts this way. From here on, **run every grey code cell when you reach it**, top to bottom. **To write in a text box** (a cell with a bold label ending in a colon), double-click it, type, then press Shift+Enter.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def keep(name):
    """The value Gemini's cell already made under this name, or ... if there is none yet."""
    global _NAMED
    v = globals().get(name, ...)
    if v is ... or v is None:
        print(f"Not there yet: nothing called {name}. Not run Gemini's green cell yet? Click Accept and "
              f"run on it, then run this cell again. Already ran it? Type in the same chat: call the "
              f"result {name}. Then Accept and run, and run this cell again.")
        _NAMED = True
        return ...
    return v


def filled(*values):
    """True once every TODO above has a value. Otherwise says what to do."""
    global _NAMED
    if any(v is ... or v is None for v in values):
        if not globals().get("_NAMED"):      # keep() has already said exactly what is missing
            print("Not there yet. Run Gemini's code first (click Accept and run on its green cell, "
                  "or paste its code into a new cell and run it), then run this cell again.")
        _NAMED = False
        return False
    _NAMED = False
    return True


def submit_prediction(team, lab, prediction, method):
    """Send one team's number to the Prediction League board."""
    import urllib.parse
    form = "https://docs.google.com/forms/d/e/1FAIpQLSdwpzHafC_rmPDdbK_cm6J8CihD-NSVS7ZBUyYCWIrS_MMJ0g/formResponse"
    fields = {'team': 'entry.911171418', 'lab': 'entry.1890998159', 'prediction': 'entry.959891957', 'method': 'entry.1997288724'}
    if not team or not method or prediction is None:
        print("Fill in TEAM, MY_PREDICTION and METHOD above, then run this cell again."); return
    entry = {"team": team, "lab": str(lab), "prediction": f"{float(prediction):.1f}", "method": method}
    if not form:
        print("The Prediction League is not switched on yet; tell your tutor your number."); return
    data = urllib.parse.urlencode({fields[k]: v for k, v in entry.items()}).encode()
    try:
        urllib.request.urlopen(urllib.request.Request(form, data=data, headers={"User-Agent": "EP0418"}), timeout=60)
        print(f"Entered: {team} predicts {float(prediction):.1f}. Check the scoreboard in a minute.")
    except Exception as e:
        print("The league did not answer:", e, "- try again in a minute.")

# The pre-lab cells need these files beside the notebook.
fetch("smart_building_sensors.csv", "smart_self_daily.csv", "smart_money_ohlcv.csv")

## Pre-lab A — Two checks your labs never used

Labs 1 to 8 ran on classroom sensors and a transport feed. Two important checks
never came up, because that data does not contain the problems they catch, but
**your project's data might**. This is the pre-lab: run it at home. If you did not,
run it now; it takes seconds.

1. **Run the cell below.**
2. **Read the report.** You should see `occupancy` PASS, `steps` NOTE, and
   `no_missing` PASS.

In [ ]:
# --- zeros_are_real: is a zero a measurement, or a device that was off? -------
# In the building data, occupancy = 0 means the room was empty. A REAL zero.
b = pd.read_csv("smart_building_sensors.csv", parse_dates=["timestamp"])
check.reset()
check.zeros_are_real(b, "occupancy", max_zeros=20000)      # passes — real zeros

# In personal activity data, steps = 0 usually means the watch was in a drawer.
# Same value, completely different meaning.
if os.path.exists("smart_self_daily.csv"):
    s = pd.read_csv("smart_self_daily.csv", parse_dates=["date"])
    check.zeros_are_real(s, "steps")                       # NOTE — investigate

# --- no_missing: are there blanks left in what you are about to analyse? ------
check.no_missing(b, cols=["co2_ppm", "temp_c"])

check.report()

### Read those two `zeros_are_real` results

`occupancy = 0` **passed**. `steps = 0` came back as a **NOTE**.

The check cannot tell which is which by itself: the cell told it, with
`max_zeros=20000` for occupancy, that many zeros are expected there.
An empty room really does have zero people in it. A person with a watch in a
drawer did not really take zero steps, and averaging those days in drags every
statistic you compute downward.

**Write in the box below** any column in your project's data where zero might mean
"not recorded", and whether you will treat its zeros as real or as missing. No such
column, or not sure of your columns yet? Write *none found* or *check when the data
is loaded*. That
line goes into the data section of your CA3 report.

**Column where zero might mean "not recorded", and what I will do with its zeros:**

## Pre-lab B — Lookahead bias

If you compute a signal from today's data and act on it today, you used
information you would not have had at the time. This is the most common way a
student backtest produces an impressive and completely fake result. It matters most
for Track 1 (Smart Money) and for any project that acts on a signal, but everyone
runs it.

1. **Run the cell below.** You should see one report with `signal_is_shifted`
   **FAIL**, then a second with it **PASS**.

In [ ]:
if os.path.exists("smart_money_ohlcv.csv"):
    m = pd.read_csv("smart_money_ohlcv.csv", parse_dates=["Date"])
    raw_signal = (m.Close > m.Close.rolling(20).mean()).astype(int)

    check.reset()
    check.signal_is_shifted(raw_signal, raw_signal, by=1)        # FAILS - lookahead
    check.report()
    check.signal_is_shifted(raw_signal, raw_signal.shift(1), by=1)  # PASSES
    check.report()

The first call fails and the second passes, on the same data. The only difference
is one `.shift(1)` — acting on yesterday's signal instead of today's.

Nothing to do here now. If your project triggers an action from a signal, run
`check.signal_is_shifted` on your own signal in your project notebook, the way this
cell does, and keep the PASS as evidence for CA3.

## Task 1 — Run the critic (in your Lab 8 notebook)

The critic needs to see your pipeline. Gemini in Colab reads the notebook it is
open in, so you run the critic **inside your Lab 8 notebook**.

1. **Copy the prompt** in the grey box below first: hover over the box and click the
   copy icon at its top right. No copy icon? Click just before *You are reviewing*,
   drag down to the end of *defined in this notebook.*, and press Ctrl+C (Cmd+C on a
   Mac). Do not press Ctrl+A: that selects the whole page. Its last line names
   the functions ingest, clean, analyse, decide, respond and pipeline: those are
   the names Lab 8 used. If you renamed any, edit that line in the chat box after
   pasting and before pressing Enter.
2. **Open your Lab 8 notebook** in a new browser tab: in Google Drive, open the
   folder *Colab Notebooks* and double-click *Copy of Lab08_End_to_End_STUB* (or
   whatever you renamed it to).
3. **Run it all:** menu **Runtime → Run all**, so your functions exist. It is done
   when every grey code cell shows a number in square brackets on its left (text
   cells never do). If a cell waits for
   a file upload, click **Cancel** under it; Colab then carries on with the rest.
   A box asking to give the notebook access to a secret? Click **Grant access**.
   Run all stopped at a red error? Click the next cell below the red one and choose
   **Runtime → Run cell and below**; the critic will still see the rest.
4. **Open Gemini** there (spark icon, bottom centre) and start a new chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**).
   A fresh chat has not seen your reasoning and has no work of its own to defend.
5. **Paste the prompt** and press Enter.
6. **Read the answer.** If Gemini offers a green cell, click **Cancel**: the critic
   reviews, it does not change your code. Ignore the blue suggestion chips.
7. **Come back to this notebook** (its browser tab) and fill in Task 2.

```
You are reviewing a student data pipeline for a module on automated decision
systems. Be adversarial but specific: your job is to find real defects, not to
be encouraging. Flag only defects that affect correctness or the checks below;
do not suggest style changes or extra features. Do not write or change code.

Check for each of these and say PASS, FAIL or UNSURE with a one-line reason:

1. Silent row loss: does any step drop rows without accounting for them?
2. Sentinel values: are non-measurement codes (-999, 0, 9999) handled?
3. Zero vs missing: are zeros that mean "no reading" distinguished from real zeros?
4. Time index: is the data on a regular grid before any .shift()?
5. Horizon mismatch: are the model and baseline scored at the same horizon?
6. Lookahead bias: does any signal use information from the future?
7. Baseline honesty: is the baseline the strongest one a sensible person would use?
8. Response layer: is the alert debounced and rate-limited?
9. Failure mode: what happens if the data source returns nothing?

For each FAIL give the specific line and what you would change.

My pipeline is the functions ingest, clean, analyse, decide, respond and
pipeline defined in this notebook.
```

## Task 2 — Record every finding, then answer it

One row per probe. **Your verdict column is what gets marked.** Use the table on
the Lab 9 sheet, or this one:

1. **Double-click the table** in the cell below (it has a cell of its own). Type
   between the `|` marks and keep the marks.
2. **Critic's finding:** fill this column for all nine rows first, before you open
   any new Gemini chat (a new chat hides the critic's answer). Copy its PASS, FAIL or UNSURE and its one-line reason. If it
   skipped a probe, write *not mentioned*.
3. **My verdict:** leave all three words for now; you choose in step 5.
4. **Get the evidence, one row at a time.** Go to the Lab 8 tab, start a new chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**), and type *Which line in this notebook answers this question:*
   followed by that row's question from the critic prompt above (for example *does
   any step drop rows without accounting for them?*), then *Quote the line and the
   printed number that shows it. End with one word: problem yes or problem no.*
   Press Enter. If Gemini adds a green cell, click **Cancel**. Do this for every row
   (a new chat each time is fine: the critic's findings are already in your table).
5. **Choose the verdict** by comparing Gemini's last word with the critic, then
   delete the other two words in that row:
   - critic said PASS and Gemini says *problem no*, or critic said FAIL and Gemini
     says *problem yes*: they agree, keep *Accept*. Evidence for a PASS: *critic
     said PASS; I agree*. Evidence for a FAIL: leave empty now; after Task 3 write
     what you changed.
   - they disagree (PASS with *problem yes*, or FAIL with *problem no*): keep
     *Reject*. Evidence: the line or number Gemini quoted.
   - Gemini quoted no line, gave no yes or no, or the critic said UNSURE or *not
     mentioned*: keep *Unsure*. Evidence: what you would need to check.
   Write one short line per cell, with no `|` in it and no line breaks.
6. **Press Shift+Enter** to see the table again.

| # | Probe | Critic's finding | My verdict | Evidence |
|---|---|---|---|---|
| 1 | Silent row loss | | Accept / Reject / Unsure | |
| 2 | Sentinel values | | Accept / Reject / Unsure | |
| 3 | Zero vs missing | | Accept / Reject / Unsure | |
| 4 | Time index | | Accept / Reject / Unsure | |
| 5 | Horizon mismatch | | Accept / Reject / Unsure | |
| 6 | Lookahead bias | | Accept / Reject / Unsure | |
| 7 | Baseline honesty | | Accept / Reject / Unsure | |
| 8 | Response layer | | Accept / Reject / Unsure | |
| 9 | Failure mode | | Accept / Reject / Unsure | |

Task 3 is only for FAILs you accepted.
**Reject** = it is wrong, and here is the evidence that it is wrong.
**Unsure** = I could not tell; here is what I would need to check. A PASS from
the critic still gets a row, filled in as step 5 says.

A row that says *Reject: I checked rows 400 to 460 and the resample keeps the empty
slots as NaN, it does not drop them* is worth more than one that says *Accept*.

## Task 3 — Fix what deserves fixing, then re-verify

Do steps 1 to 3 once for each **FAIL** finding you **accepted** (not the PASSes you agreed with),
then steps 4 and 5 once at the end:

1. **Copy the prompt below** here in the Lab 9 tab: drag from *Fix this* to *existing
   check.* and press Ctrl+C (Cmd+C on a Mac). Switch to the Lab 8 tab, in Gemini start a new chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**), and paste it (Ctrl+V). In the chat box, delete *[the
   finding, copied from the critic]* including the square brackets, and type the
   finding there in a few words, from the *Critic's finding* column of your Task 2 table (for
   example *clean() drops the outage rows without counting them*). Press Enter:

   > Fix this: [the finding, copied from the critic]. Then write a test that shows
   > the fix works: print the row count before and after, and the smallest and
   > largest value of the columns it touches. Do not change any existing check.

2. **Read the green cell's code before accepting.** Does any line start with
   `check.`, or does the answer tell you to delete one? Usually no: click **Accept
   and run**. If yes, click **Cancel** and type in the same chat *Do the same fix
   without changing or deleting any check. line*, then read the new green cell the
   same way. Remember your answer for the box in step 3.
3. **Judge its test, not only its answer.** Back in this notebook, in the box below,
   for each fix, answer: what did the test actually check (*"it returns a table"*
   proves nothing)? Did it touch a `check.` line (your note from step 2)? Did it show
   numbers, or only say "passed" (a pass with no numbers is a claim, not evidence)?
4. **Run your Lab 8 Task 3 cells again**: first the cell that prints rows removed and
   added, then the check cell under it (the one with `EXPLANATION =`). If the check
   prints FAIL for `rows_accounted_for`, the numbers no longer match your
   explanation: double-click inside that check cell, replace the text between the quotes after
   `EXPLANATION =` with the new numbers, written the same way as in Lab 8 Task 3
   step 1 (number, removed or added, reason; joined with semicolons), and run the
   check cell again. Any other FAIL, or a red error: the fix broke something. Write that in the
   box below, put your hand up, and show the tutor. Copy the PASS and
   FAIL lines into the box.
5. **Go back to the Task 2 table** and write what you changed in the Evidence
   column of each FAIL you accepted.

Accepted nothing? Write *no fixes* and why, and move on.

**Each fix, and what its test really showed:**

**My Lab 8 checks after the fixes (copy the PASS / FAIL lines):**

## The question the critic cannot answer

An AI critic can check your code. It cannot tell you **what happens on the days
your system is wrong, and who pays for it.** That question is the Week 14 record
box on your CA3 Build 3 sheet, and it goes straight into your CA3 reflection. You
write it next week; for now, note one idea in the *Decision* box at the end.

---
## Read before you run

A Track 5 group asked the critic to check their weekly activity alert. The critic said PASS on “zero versus missing”. The code is below; its last lines print the count both ways.

The code is in the cell below, commented out. It came from an AI assistant in
someone else's chat; you are not asking Gemini for it.

1. **Read it. Do not run it.** Answer questions 1 and 2 (just below) in the text
   box under the code cell (double-click it to type), or on the sheet.
2. **Ask Gemini to explain it.** Click inside the code cell, press Ctrl+A (Cmd+A on
   a Mac; inside a code cell this selects only that cell) and Ctrl+C to copy it.
   Open Gemini (spark icon, bottom centre), start a new chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**), paste the
   code into the chat box, then type a space and *Explain this code in plain words.*
   after it on the same line. Press Enter once, to send. If Gemini adds a green cell,
   click **Cancel**: this step only explains. Read the explanation. Did it point out a problem with the code, the one your questions are about? It often does not.
3. **Only then run it.** Delete the `# ` at the start of every line *below the
   first line* (the first line is the instruction and stays), then run the cell
   and see whether you were right.

**Question 1.** Was the critic right? Eleven days in this file have `steps` equal to 0 because the watch was not worn. What do those days do to `weekly`, and to `their_df.steps.mean()`?

**Question 2.** Predict which of the two printed counts is larger, and why. At step 3, run it and compare.

<details><summary>Hint 1: where to look</summary>

Inside `rolling().mean()`, is a 0 treated like a NaN, or like a reading?

</details>
<details><summary>Hint 2: closer to the answer</summary>

Like a reading. A drawer day pulls the weekly average and the overall mean down. The two printed counts show how much the zeros matter.

</details>

An explanation that reads well is not evidence the code is right; that is the
point of step 2. 

**Optional, to fix it:** in the same Gemini chat, type *Fix the problem in this
code that Hint 2 describes. Say in one sentence what you changed.* (Gemini can read
the hint in the notebook; you can open it yourself by clicking its title.) Press Enter. If Gemini's sentence describes one change, and it matches
what Hint 2 says is wrong, click **Accept and run**. If it lists other changes too,
click **Cancel** and ask again with *fix only that, nothing else*.

In [ ]:
# ---- Every line below this one is the code. Do steps 1 and 2 first. For step 3, delete the "# " at the start of each line below, then run the cell. ----
# their_df = pd.read_csv("smart_self_daily.csv", parse_dates=["date"])
# weekly = their_df.steps.rolling(7).mean()
# low_activity = weekly < 0.7 * their_df.steps.mean()
# print("days flagged, zeros left in       :", int(low_activity.sum()))
# st = their_df.steps.replace(0, float("nan"))
# print("days flagged, zeros treated as NaN:", int((st.rolling(7, min_periods=4).mean() < 0.7 * st.mean()).sum()))

**Answer to question 1:**

**Answer to question 2:**

**Did Gemini's explanation mention the fault? (yes / no, and the sentence):**

**Was I right, once I ran it?**

---
## Quiz yourself before you leave (5 minutes)

The checks tell you whether the code is right. They do not tell you whether
**you** know what it did. In this notebook, open Gemini (spark icon, bottom centre)
and start a new chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**). Copy the quoted prompt below: drag from its first
word to its last and press Ctrl+C (Cmd+C on a Mac). Paste it into the chat box and
press Enter:

> Quiz me on this lab. Ask five short questions, one at a time, in plain words, on these ideas: what an adversarial review is for; why a critic's PASS is not evidence; how to defend correct work against a wrong finding; why you run the critic in a new chat, not the one that wrote the code; the question the critic cannot answer: who pays when the system is wrong. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks in the chat panel; you answer in the chat box; it marks you and asks
the next. Ignore the suggestion chips under its messages. Answer in your own
words, without looking back until it has marked you.

Then write your score and the idea to revise below. Your tutor asks for them in
the plenary.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

---
## Before you leave

Double-click the box cell below and type there, or write on the last page of the Lab 9
sheet. The findings table must have a verdict and evidence on **every** row.

- **Mistake caught:** in the box below, write the critic's mistake from a finding
  you rejected, in the box's three parts: what it claimed · what
  was really there · how you showed it. Rejected none? Use a finding you were unsure
  about instead: what it claimed · what you could not tell · what you would check.
- **Decision:** which findings you fixed and why, and one idea about who pays when
  your system is wrong.

**Reality check:** if you accepted every finding without pushing back on any, you
probably have not read them closely enough. Critics are confidently wrong too:
that is the whole lesson of this module.

**Mistake caught:** *(what the critic claimed · what was really there · how I showed it. Rejected none? A finding I was unsure about · what I could not tell · what I would check.)*

**Decision:** *(what I decided · why · the evidence)*